<a href="https://colab.research.google.com/github/joaocanaslopes/Assignments_ML/blob/main/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Petrignano.csv to Petrignano.csv


In [7]:
df = pd.read_csv('Petrignano.csv')
display(df.head(10))

,Date,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano
0,1/1/2009,0.0,-31.14,5.2,4.9,-24530.688,2.4
1,2/1/2009,0.0,-31.11,2.3,2.5,-28785.888,2.5
2,3/1/2009,0.0,-31.07,4.4,3.9,-25766.208,2.4
3,4/1/2009,0.0,-31.05,0.8,0.8,-27919.296,2.4
4,5/1/2009,0.0,-31.01,-1.9,-2.1,-29854.656,2.3
5,6/1/2009,0.0,-31.00,-0.7,-0.7,-29124.576,2.3
6,7/1/2009,0.0,-30.96,1.5,-0.3,-31173.120,2.3
7,8/1/2009,0.0,-30.94,4.3,6.6,-30232.224,2.4
8,9/1/2009,0.9,-30.93,4.9,4.8,-30597.696,2.3
9,10/1/2009,0.0,-30.87,1.9,4.2,-31337.280,2.3


### Processamento da Coluna 'Date'

Este bloco de código converte a coluna 'Date' para o formato `datetime`, define-a como o índice do DataFrame e garante que o DataFrame esteja ordenado cronologicamente. Uma verificação é incluída para evitar erros caso a coluna 'Date' já tenha sido processada.

In [16]:
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
df = df.set_index('Date')
df = df.sort_index()
display(df.head())

A coluna 'Date' já é o índice ou não está presente como coluna. O DataFrame provavelmente já foi processado.


,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano
Date,,,,,,
2009-01-01,0.0,-31.14,5.2,4.9,-24530.688,2.4
2009-01-02,0.0,-31.11,2.3,2.5,-28785.888,2.5
2009-01-03,0.0,-31.07,4.4,3.9,-25766.208,2.4
2009-01-04,0.0,-31.05,0.8,0.8,-27919.296,2.4
2009-01-05,0.0,-31.01,-1.9,-2.1,-29854.656,2.3


In [19]:
pd.set_option('display.max_rows', None)
missing_values_per_row = df.isnull().sum(axis=1)
display(missing_values_per_row)
pd.reset_option('display.max_rows')

,0
Date,
2009-01-01,0
2009-01-02,0
2009-01-03,0
2009-01-04,0
2009-01-05,0
2009-01-06,0
2009-01-07,0
2009-01-08,0
2009-01-09,0


In [21]:
df.ffill(inplace=True)

In [23]:
df['month'] = df.index.month
display(df.head(70))

,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano,month
Date,,,,,,,
2009-01-01,0.0,-31.14,5.2,4.9,-24530.688,2.4,1
2009-01-02,0.0,-31.11,2.3,2.5,-28785.888,2.5,1
2009-01-03,0.0,-31.07,4.4,3.9,-25766.208,2.4,1
2009-01-04,0.0,-31.05,0.8,0.8,-27919.296,2.4,1
2009-01-05,0.0,-31.01,-1.9,-2.1,-29854.656,2.3,1
...,...,...,...,...,...,...,...
2009-03-07,0.0,-28.76,9.4,9.1,-30612.384,2.6,3
2009-03-08,0.0,-28.77,9.3,8.6,-28054.080,2.8,3
2009-03-09,0.5,-28.71,9.7,9.0,-27016.416,2.5,3


In [25]:
y = df['Depth_to_Groundwater_P25']

X = df.copy()

# Calculate monthly lags for the 'month' column explicitly, based on the current month
current_month_values = X['month']
X['month_lag1'] = (current_month_values - 2) % 12 + 1 # For Jan (1), this becomes 12
X['month_lag2'] = (current_month_values - 3) % 12 + 1 # For Jan (1), this becomes 11

# Columns for which to create daily lags (all original columns except 'month')
cols_for_daily_lags = [col for col in df.columns if col != 'month']

# Create daily lagged features for other columns
for col in cols_for_daily_lags:
    X[f'{col}_lag1'] = X[col].shift(1)
    X[f'{col}_lag2'] = X[col].shift(2)

# Drop the original columns from X, including the original 'month' from df.columns
X = X.drop(columns=df.columns)

# Align y with X after creating lagged features
y_aligned = y.loc[X.index]

# Combine X and y for easier NaN handling
df_combined = pd.concat([X, y_aligned], axis=1)

# Drop rows with NaN values resulting from lags (mainly from the daily shifts of other features)
df_combined.dropna(inplace=True)

# Separate X and y again after dropping NaNs
X = df_combined.drop(columns=['Depth_to_Groundwater_P25'])
y = df_combined['Depth_to_Groundwater_P25']

print("Variáveis preditoras (X) e variável resposta (y) criadas com sucesso.")
print(f"Formato de X: {X.shape}")
print(f"Formato de y: {y.shape}")
display(X.head())
display(y.head())

Variáveis preditoras (X) e variável resposta (y) criadas com sucesso.
Formato de X: (4197, 14)
Formato de y: (4197,)


,month_lag1,month_lag2,Rainfall_Bastia_Umbra_lag1,Rainfall_Bastia_Umbra_lag2,Depth_to_Groundwater_P25_lag1,Depth_to_Groundwater_P25_lag2,Temperature_Bastia_Umbra_lag1,Temperature_Bastia_Umbra_lag2,Temperature_Petrignano_lag1,Temperature_Petrignano_lag2,Volume_C10_Petrignano_lag1,Volume_C10_Petrignano_lag2,Hydrometry_Fiume_Chiascio_Petrignano_lag1,Hydrometry_Fiume_Chiascio_Petrignano_lag2
Date,,,,,,,,,,,,,,
2009-01-03,12,11,0.0,0.0,-31.11,-31.14,2.3,5.2,2.5,4.9,-28785.888,-24530.688,2.5,2.4
2009-01-04,12,11,0.0,0.0,-31.07,-31.11,4.4,2.3,3.9,2.5,-25766.208,-28785.888,2.4,2.5
2009-01-05,12,11,0.0,0.0,-31.05,-31.07,0.8,4.4,0.8,3.9,-27919.296,-25766.208,2.4,2.4
2009-01-06,12,11,0.0,0.0,-31.01,-31.05,-1.9,0.8,-2.1,0.8,-29854.656,-27919.296,2.3,2.4
2009-01-07,12,11,0.0,0.0,-31.00,-31.01,-0.7,-1.9,-0.7,-2.1,-29124.576,-29854.656,2.3,2.3


,Depth_to_Groundwater_P25
Date,
2009-01-03,-31.07
2009-01-04,-31.05
2009-01-05,-31.01
2009-01-06,-31.00
2009-01-07,-30.96


In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Formato de X_train: {X_train.shape}")
print(f"Formato de X_test: {X_test.shape}")
print(f"Formato de y_train: {y_train.shape}")
print(f"Formato de y_test: {y_test.shape}")

Formato de X_train: (3357, 14)
Formato de X_test: (840, 14)
Formato de y_train: (3357,)
Formato de y_test: (840,)


In [28]:
cv_naive = KFold(n_splits=5, shuffle=True, random_state=42)
cv_temporal = TimeSeriesSplit(n_splits=5)

In [30]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('dt_regressor', DecisionTreeRegressor(max_depth=10, random_state=42))
])

print(pipeline)

Pipeline(steps=[('scaler', StandardScaler()),
                ('dt_regressor',
                 DecisionTreeRegressor(max_depth=10, random_state=42))])


In [31]:
# Avaliar o pipeline com cv_naive
r2_scores_naive = cross_val_score(pipeline, X_train, y_train, cv=cv_naive, scoring='r2')
print(f"R2 médio com KFold (shuffle=True): {r2_scores_naive.mean():.4f}")

# Avaliar o pipeline com cv_temporal
r2_scores_temporal = cross_val_score(pipeline, X_train, y_train, cv=cv_temporal, scoring='r2')
print(f"R2 médio com TimeSeriesSplit: {r2_scores_temporal.mean():.4f}")

R2 médio com KFold (shuffle=True): 0.9994
R2 médio com TimeSeriesSplit: 0.6089


In [33]:
pipeline.fit(X_train, y_train)
y_pred_test = pipeline.predict(X_test)
r2_test = r2_score(y_test, y_pred_test)

print(f"R2 no conjunto de teste final: {r2_test:.4f}")

R2 no conjunto de teste final: 0.9858


In [34]:
def evaluate_model_selection(X_train, y_train, X_test, y_test, cv_strategy, name):
    print(f"\n--- Avaliação com {name} ---")

    # Pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', DecisionTreeRegressor(random_state=42))
    ])

    # param_grid com pelo menos 4 profundidades
    param_grid = {
        'regressor__max_depth': [5, 10, 15, 20, 25]
    }

    # GridSearchCV
    grid_search = GridSearchCV(pipeline, param_grid, cv=cv_strategy, scoring='r2', n_jobs=-1, verbose=1)
    grid_search.fit(X_train, y_train)

    # Imprimir os melhores parâmetros e R2 da validação interna
    print(f"Melhores parâmetros: {grid_search.best_params_}")
    print(f"R2 da validação interna: {grid_search.best_score_:.4f}")

    # R2 do teste independente
    y_pred_test = grid_search.predict(X_test)
    r2_test_independent = r2_score(y_test, y_pred_test)
    print(f"R2 do teste independente: {r2_test_independent:.4f}")

    return grid_search

# Executa a função para as duas estratégias
result_naive = evaluate_model_selection(X_train, y_train, X_test, y_test, cv_naive, 'Naive KFold (shuffle=True)')
result_temporal = evaluate_model_selection(X_train, y_train, X_test, y_test, cv_temporal, 'Temporal TimeSeriesSplit')



--- Avaliação com Naive KFold (shuffle=True) ---
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Melhores parâmetros: {'regressor__max_depth': 10}
R2 da validação interna: 0.9994
R2 do teste independente: 0.9858

--- Avaliação com Temporal TimeSeriesSplit ---
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Melhores parâmetros: {'regressor__max_depth': 25}
R2 da validação interna: 0.6139
R2 do teste independente: 0.9813
